# Synthetic Detector Analysis

This notebook uses synthetic data to illustrate detector calibration, assignment-level summaries, and detector-conflict review for a private research toolkit. It does not access private files, cloud storage, raw submissions, or participant records.

## Scope

This notebook uses synthetic data only:

1. Calibrate detector behavior against known synthetic categories.
2. Combine detector scores, summarize them by group, and plot distributions.
3. Flag files where detectors disagree on labels or have large score spreads.

All numerical outputs are generated randomly in this notebook and are illustrative, not research findings.

In [ ]:
from __future__ import annotations

from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Fixed random seed for reproducible synthetic example generation
SEED = 7
rng = np.random.default_rng(SEED)

# Tools under evaluation and their display labels
TOOLS = ["gptzero", "pangram", "sapling", "isgen"]
TOOL_LABELS = {
    "gptzero": "GPTZero",
    "pangram": "Pangram",
    "sapling": "Sapling",
    "isgen": "Isgen",
}

# Mapping of synthetic reference set names to their intended source label
# (used when creating or interpreting synthetic examples)
REFERENCE_SETS = {
    "ai_free": "HUMAN",
    "humanized": "AI",
    "ai_full": "AI",
}

# Directory for generated synthetic examples; created if missing
OUTPUT_DIR = Path("synthetic_examples/generated")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Pandas display settings for interactive inspection
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

## Synthetic calibration sample

Create a synthetic reference table with three known categories:

- `ai_free` → `HUMAN`
- `humanized` → `AI`
- `ai_full` → `AI`

The generator makes `humanized` harder to classify than the other categories to motivate weighting logic. This is a synthetic design choice, not an observed property of real data.

In [ ]:
def clip01(x):
    # Clamp numeric rate to the allowed [0, 100] interval and return as float
    return float(np.clip(x, 0.0, 100.0))


def label_from_rate(rate: float) -> str:
    # Simple thresholding: <=30 => HUMAN, >=70 => AI, otherwise MIXED
    if rate >= 70:
        return "AI"
    if rate <= 30:
        return "HUMAN"
    return "MIXED"


# Calibration profiles: (mean_percent, sd_percent) for synthetic AI-rate generation
calibration_profiles = {
    "gptzero": {"ai_free": (6, 5), "humanized": (76, 14), "ai_full": (95, 5)},
    "pangram": {"ai_free": (4, 4), "humanized": (68, 18), "ai_full": (97, 4)},
    "sapling": {"ai_free": (14, 10), "humanized": (63, 16), "ai_full": (92, 7)},
    "isgen": {"ai_free": (10, 8), "humanized": (42, 18), "ai_full": (84, 10)},
}

# Build synthetic dataset of reference files. Each entry contains per-tool, per-version
# simulated AI-rate (normal distribution, then clipped) and a categorical label.
rows = []
n_reference_files = 40
for i in range(n_reference_files):
    row = {"file_id": f"ref_{i+1:03d}"}
    for version in REFERENCE_SETS:
        for tool in TOOLS:
            mean, sd = calibration_profiles[tool][version]
            # Draw from a normal distribution, then clamp to [0,100]
            rate = clip01(rng.normal(mean, sd))
            row[f"{version}_{tool}_ai_rate_percent"] = rate
            row[f"{version}_{tool}_main_result"] = label_from_rate(rate)
    rows.append(row)

calibration_df = pd.DataFrame(rows)
calibration_df.head()

In [ ]:
def normalize_label(value) -> str:
    if pd.isna(value):
        return "UNKNOWN"
    s = str(value).strip().lower()
    if "error" in s:
        return "ERROR"
    if "mixed" in s:
        return "MIXED"
    if "human" in s and "ai" not in s:
        return "HUMAN"
    if "ai" in s:
        return "AI"
    return "UNKNOWN"


def pct(n, d):
    return round(100 * n / d, 2) if d else 0.0


calibration_rows = []
for tool in TOOLS:
    for version, expected in REFERENCE_SETS.items():
        result_col = f"{version}_{tool}_main_result"
        rate_col = f"{version}_{tool}_ai_rate_percent"
        predicted = calibration_df[result_col].map(normalize_label)
        correct = int((predicted == expected).sum())
        total = int(len(predicted))
        calibration_rows.append(
            {
                "tool": TOOL_LABELS[tool],
                "reference_set": version,
                "expected_label": expected,
                "accuracy_percent": pct(correct, total),
                "avg_ai_rate_percent": round(float(calibration_df[rate_col].mean()), 2),
            }
        )

calibration_summary = pd.DataFrame(calibration_rows)
calibration_summary

In [ ]:
## Pivot calibration_summary to show accuracy (%) by tool and reference set
accuracy_pivot = calibration_summary.pivot(
    index="tool", columns="reference_set", values="accuracy_percent"
)

# Pivot calibration_summary to show average AI rate (%) by tool and reference set
rate_pivot = calibration_summary.pivot(
    index="tool", columns="reference_set", values="avg_ai_rate_percent"
)

# Display tables for inspection; expects calibration_summary to contain one row per tool/reference_set
display(accuracy_pivot)
display(rate_pivot)

In [ ]:
overall_tool_accuracy = (
    calibration_summary.groupby("tool", as_index=False)["accuracy_percent"]
    .mean()
    .rename(columns={"accuracy_percent": "overall_accuracy_percent"})
)
overall_tool_accuracy

The calibration table is synthetic. In production, derive these weights from a documented calibration dataset rather than hard-coding them.

## 2. Synthetic assignment-level dataset

Create a synthetic assignments table with `University`, `Major`, `Year`, `Language`, `Assignment Type`, and `word_count`. Generate detector scores using simple heuristics to produce non-trivial downstream summaries. The data are fully synthetic.

In [ ]:
universities = ["North", "Central", "Coastal", "Metro"]
majors = ["CS", "Math", "Engineering", "Humanities"]
years = ["Year 1", "Year 2", "Year 3", "Year 4"]
languages = ["English", "Arabic", "Code"]
assignment_types = ["Essay", "Report", "Lab", "Project", "Code Review"]

language_base = {"English": 52, "Arabic": 66, "Code": 58}
type_base = {"Essay": 62, "Report": 55, "Lab": 48, "Project": 57, "Code Review": 58}
major_shift = {"CS": 4, "Math": -2, "Engineering": 1, "Humanities": 3}

tool_bias = {"gptzero": -6, "pangram": 4, "sapling": 10, "isgen": -10}
tool_noise = {"gptzero": 18, "pangram": 16, "sapling": 14, "isgen": 20}


def synthetic_word_count(language: str, assignment_type: str) -> int:
    if language == "Code":
        return int(np.clip(rng.normal(1800, 500), 200, 5000))
    if assignment_type == "Lab":
        return int(np.clip(rng.normal(2600, 700), 300, 7000))
    if assignment_type == "Essay":
        return int(np.clip(rng.normal(1200, 350), 200, 4000))
    return int(np.clip(rng.normal(2200, 650), 300, 7000))


def synthetic_detector_rate(language, assignment_type, major, word_count, tool):
    base = (
        language_base[language] + type_base[assignment_type] - 55 + major_shift[major]
    )
    length_effect = -0.006 * max(word_count - 1200, 0)
    rate = base + tool_bias[tool] + length_effect + rng.normal(0, tool_noise[tool])
    return clip01(rate)


n_files = 180
assignment_rows = []
for i in range(n_files):
    language = rng.choice(languages, p=[0.5, 0.3, 0.2])
    assignment_type = rng.choice(assignment_types, p=[0.25, 0.25, 0.15, 0.2, 0.15])
    if language == "Code":
        assignment_type = "Code Review"

    major = rng.choice(majors)
    university = rng.choice(universities)
    year = rng.choice(years)
    word_count = synthetic_word_count(language, assignment_type)

    row = {
        "file_name": f"synthetic_{i+1:03d}.txt",
        "University": university,
        "Major": major,
        "Year": year,
        "Language": language,
        "Assignment Type": assignment_type,
        "word_count": word_count,
    }

    for tool in TOOLS:
        if language == "Code" and tool != "pangram":
            row[f"{tool}_ai_rate_percent"] = np.nan
            row[f"{tool}_result"] = np.nan
            continue

        rate = synthetic_detector_rate(
            language, assignment_type, major, word_count, tool
        )
        row[f"{tool}_ai_rate_percent"] = rate
        row[f"{tool}_result"] = label_from_rate(rate)

    assignment_rows.append(row)

assignments_df = pd.DataFrame(assignment_rows)
assignments_df.head()

In [ ]:
raw_weights = {
    row["tool"].lower(): row["overall_accuracy_percent"]
    for _, row in overall_tool_accuracy.iterrows()
}
weight_sum = sum(raw_weights.values())
norm_weights = {tool: value / weight_sum for tool, value in raw_weights.items()}
norm_weights

In [ ]:
def is_coding_assignment(row) -> bool:
    return row["Language"] == "Code" or "code" in str(row["Assignment Type"]).lower()


def weighted_combined_ai_rate(row) -> float:
    if is_coding_assignment(row):
        return row["pangram_ai_rate_percent"]

    available = {
        tool: row[f"{tool}_ai_rate_percent"]
        for tool in TOOLS
        if pd.notna(row.get(f"{tool}_ai_rate_percent"))
    }
    if not available:
        return np.nan

    usable_weight_sum = sum(norm_weights[tool] for tool in available)
    return (
        sum(available[tool] * norm_weights[tool] for tool in available)
        / usable_weight_sum
    )


assignments_df["combined_ai_rate_percent"] = assignments_df.apply(
    weighted_combined_ai_rate, axis=1
)
assignments_df["combined_label_50"] = np.where(
    assignments_df["combined_ai_rate_percent"] >= 50, "AI", "HUMAN"
)

assignments_df[
    [
        "file_name",
        "Language",
        "Assignment Type",
        "combined_ai_rate_percent",
        "combined_label_50",
    ]
].head()

In [ ]:
def summarize_series(series: pd.Series) -> dict:
    s = pd.to_numeric(series, errors="coerce").dropna()
    return {
        "n": int(s.shape[0]),
        "mean": round(float(s.mean()), 2),
        "median": round(float(s.median()), 2),
        "sd": round(float(s.std(ddof=1)), 2) if len(s) > 1 else np.nan,
        "min": round(float(s.min()), 2),
        "max": round(float(s.max()), 2),
    }


overall_descriptives = pd.DataFrame(
    {
        "Combined": summarize_series(assignments_df["combined_ai_rate_percent"]),
        "GPTZero": summarize_series(assignments_df["gptzero_ai_rate_percent"]),
        "Pangram": summarize_series(assignments_df["pangram_ai_rate_percent"]),
        "Sapling": summarize_series(assignments_df["sapling_ai_rate_percent"]),
        "Isgen": summarize_series(assignments_df["isgen_ai_rate_percent"]),
    }
).T
overall_descriptives

In [ ]:
def grouped_summary_table(data: pd.DataFrame, group_col: str) -> pd.DataFrame:
    out = (
        data.groupby(group_col)
        .agg(
            n_files=("file_name", "count"),
            mean_combined_ai_rate=("combined_ai_rate_percent", "mean"),
            median_combined_ai_rate=("combined_ai_rate_percent", "median"),
            sd_combined_ai_rate=("combined_ai_rate_percent", "std"),
            mean_word_count=("word_count", "mean"),
        )
        .reset_index()
        .sort_values(["mean_combined_ai_rate", "n_files"], ascending=[False, False])
    )
    numeric_cols = [c for c in out.columns if c not in {group_col, "n_files"}]
    out[numeric_cols] = out[numeric_cols].round(2)
    return out


for group_col in ["University", "Major", "Year", "Language", "Assignment Type"]:
    print(f"\n=== {group_col} ===")
    display(grouped_summary_table(assignments_df, group_col))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Expect grouped_summary_table to return a DataFrame with columns:
#   - 'Language'
#   - 'mean_combined_ai_rate' (numeric, already in percent units or comparable scale)
# This bar plot shows mean AI-rate per language; interpretation depends on how
# mean_combined_ai_rate was computed (e.g., weighted/unweighted by assignment counts).
language_table = grouped_summary_table(assignments_df, "Language")
axes[0].bar(
    language_table["Language"],
    language_table["mean_combined_ai_rate"],
    color="steelblue",
)
axes[0].set_title("Mean combined AI-rate by language")
axes[0].set_ylabel("AI-rate (%)")

# Scatter of per-assignment word count vs combined AI-rate (percent).
# Each point is one assignment; overlapping points may obscure density.
# combined_ai_rate_percent must be on a percent scale for the labeled axis.
axes[1].scatter(
    assignments_df["word_count"],
    assignments_df["combined_ai_rate_percent"],
    alpha=0.6,
    s=20,
)
axes[1].set_title("Word count vs combined AI-rate")
axes[1].set_xlabel("word_count")
axes[1].set_ylabel("combined_ai_rate_percent")

plt.tight_layout()
plt.show()

## 3. Detector conflicts

Using synthetic tables only, flag files when:

- **Label conflict:** available detectors map to different coarse labels.
- **Rate conflict:** the difference between the highest and lowest detector score exceeds a threshold.

Source documents are not copied or inspected.

In [ ]:
CONFLICT_RATE_RANGE_THRESHOLD = 40.0

for tool in TOOLS:
    assignments_df[f"{tool}_label_norm"] = assignments_df[f"{tool}_result"].map(
        normalize_label
    )


def available_labels(row):
    labels = []
    for tool in TOOLS:
        value = row.get(f"{tool}_label_norm")
        if pd.notna(value) and value not in {"ERROR", "UNKNOWN"}:
            labels.append(value)
    return labels


def available_rates(row):
    rates = []
    for tool in TOOLS:
        value = row.get(f"{tool}_ai_rate_percent")
        if pd.notna(value):
            rates.append(float(value))
    return rates


def has_label_conflict(row) -> bool:
    return len(set(available_labels(row))) >= 2


def has_rate_conflict(row, threshold=CONFLICT_RATE_RANGE_THRESHOLD) -> bool:
    rates = available_rates(row)
    return len(rates) >= 2 and (max(rates) - min(rates)) >= threshold


assignments_df["label_conflict"] = assignments_df.apply(has_label_conflict, axis=1)
assignments_df["rate_conflict"] = assignments_df.apply(has_rate_conflict, axis=1)
assignments_df["any_conflict"] = (
    assignments_df["label_conflict"] | assignments_df["rate_conflict"]
)
assignments_df["rate_range"] = assignments_df.apply(
    lambda row: (
        max(available_rates(row)) - min(available_rates(row))
        if len(available_rates(row)) >= 2
        else np.nan
    ),
    axis=1,
)

conflict_df = assignments_df.loc[assignments_df["any_conflict"]].copy()
conflict_df[
    [
        "file_name",
        "Language",
        "Assignment Type",
        "combined_ai_rate_percent",
        "label_conflict",
        "rate_conflict",
        "rate_range",
    ]
].head(15)

In [ ]:
conflict_summary = pd.DataFrame(
    {
        "n_total_files": [len(assignments_df)],
        "n_conflicting_files": [int(assignments_df["any_conflict"].sum())],
        "pct_conflicting_files": [
            round(100 * assignments_df["any_conflict"].mean(), 2)
        ],
        "mean_rate_range": [
            round(float(assignments_df["rate_range"].dropna().mean()), 2)
        ],
    }
)
conflict_summary

In [ ]:
pairwise_rows = []
for left, right in combinations(TOOLS, 2):
    left_col = f"{left}_ai_rate_percent"
    right_col = f"{right}_ai_rate_percent"
    sub = assignments_df[[left_col, right_col]].dropna()
    diffs = (sub[left_col] - sub[right_col]).abs()
    pairwise_rows.append(
        {
            "pair": f"{left} vs {right}",
            "count": int(len(diffs)),
            "mean_abs_diff": round(float(diffs.mean()), 2),
            "median_abs_diff": round(float(diffs.median()), 2),
            "max_abs_diff": round(float(diffs.max()), 2),
        }
    )

pairwise_diff_summary = pd.DataFrame(pairwise_rows).sort_values(
    "mean_abs_diff", ascending=False
)
pairwise_diff_summary

In [ ]:
language_conflict_table = (
    assignments_df.groupby("Language")
    .agg(
        n_files=("file_name", "count"),
        mean_rate_range=("rate_range", "mean"),
        pct_any_conflict=("any_conflict", lambda s: 100 * s.mean()),
    )
    .reset_index()
    .round(2)
)
language_conflict_table

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_df = language_conflict_table.sort_values("pct_any_conflict", ascending=False)
ax.bar(plot_df["Language"], plot_df["pct_any_conflict"], color="indianred")
ax.set_title("Conflict rate by language (synthetic)")
ax.set_ylabel("Percent of files flagged")
plt.tight_layout()
plt.show()

## Optional export

The next cell writes synthetic CSV files containing only examples generated in this notebook. They mirror intermediate artifacts from a private pipeline.

In [ ]:
calibration_summary.to_csv(
    OUTPUT_DIR / "synthetic_calibration_summary.csv", index=False
)
assignments_df.to_csv(
    OUTPUT_DIR / "synthetic_assignments_with_combined_scores.csv", index=False
)
conflict_df.to_csv(OUTPUT_DIR / "synthetic_conflicting_files.csv", index=False)
pairwise_diff_summary.to_csv(
    OUTPUT_DIR / "synthetic_pairwise_detector_differences.csv", index=False
)

print(f"Wrote synthetic outputs to: {OUTPUT_DIR}")

Keep raw private data (submissions, identifiers, source documents) out of the repository; read file paths from configuration. Publish only schemas, synthetic examples, and pipeline code. This notebook operates on synthetic data to avoid using private corpora or course submissions.